## Setup and MLflow Initialization
In this section I initialize the environment and connect the project to DagsHub and MLflow. This allows us to track every experiment remotely ensuring that our model parameters and metrics are saved in a professional cloud registry.

In [185]:
import dagshub
import mlflow
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_squared_log_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor, AdaBoostRegressor


# Set Up Tracking Environment and Remote Server
os.environ['MLFLOW_TRACKING_USERNAME'] = 'GigiSichinava'
dagshub.init(repo_owner='GigiSichinava', repo_name='ML-Assignment-1')
mlflow.set_tracking_uri("https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow")
mlflow.set_experiment("House_Price_Regression")


Initialized MLflow to track repo "GigiSichinava/ML-Assignment-1"

Repository GigiSichinava/ML-Assignment-1 initialized!

<Experiment: artifact_location='mlflow-artifacts:/96e2751e965f44f1831b528fcd97e517', creation_time=1775950867574, experiment_id='1', last_update_time=1775950867574, lifecycle_stage='active', name='House_Price_Regression', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

## Read The Data
Selecting Data for Test and Train


In [186]:
# Data Loading and Preparation
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# Select only numeric columns and drop 'ID'
# numeric_cols = train_df.select_dtypes(include=[np.number]).columns


## Helper Function: train_and_log
This utility function trains any scikit-learn compatible model evaluates it on the validation set logs all metrics (RMSLE, RMSE, MAE, R²) to MLflow and registers the model to the DagsHub Model Registry. It is reused for every experiment below.

In [187]:
# The Metric Machine Function
def train_and_log(model, model_name):
    with mlflow.start_run(run_name=model_name):
        model.fit(X_train, y_train)
        preds = model.predict(X_val)

        # Metrics
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        mae = mean_absolute_error(y_val, preds)
        r2 = r2_score(y_val, preds)
        rmsle = np.sqrt(mean_squared_log_error(y_val, np.clip(preds, 0, None)))

        # Log Metrics
        mlflow.log_param("model_type", model_name)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2_score", r2)
        mlflow.log_metric("rmsle", rmsle)

        # Save to Model Registry
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model",
            registered_model_name=model_name
        )

        # Printout
        print(f"Model logged: {model_name}")
        print(f"   RMSLE:          {rmsle:.4f}")
        print(f"   RMSE:           ${rmse:,.0f}")
        print(f"   MAE:            ${mae:,.0f}")
        print(f"   R2 Score:       {r2:.4f}")
        print("-" * 30)

        return model

## Data Cleaning & Feature Engineering
I load the Kaggle dataset and apply the following preprocessing steps:
- **NaN Handling:** All missing numeric values are filled with `0`. This is logically sound for features like `GarageArea` or `TotalBsmtSF` where a missing value means the feature does not exist on that property.
- **Feature Type Filtering:** Only numeric columns are retained. Categorical columns are excluded in this baseline to avoid noise and data leakage from improper encoding.
- **Column Removal:** The `Id` column is dropped as it carries no predictive information and would cause overfitting.

In [188]:
def preprocess_features(train_df: pd.DataFrame, test_df: pd.DataFrame, *, corr_threshold=0.01):
    train = train_df.copy()
    test = test_df.copy()

    y = train["SalePrice"].copy()
    train_features = train.drop(columns=["SalePrice"], errors="ignore")

    # Combine for consistent encoding
    all_features = pd.concat([train_features, test], axis=0, ignore_index=True)

    # Smart NaN handling
    none_like_cats = [
        "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
        "GarageType", "GarageFinish", "GarageQual", "GarageCond",
        "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
        "MasVnrType",
    ]
    for c in none_like_cats:
        if c in all_features.columns:
            all_features[c] = all_features[c].astype("object").fillna("None")

    # LotFrontage: neighborhood-wise median (fallback to global median)
    if "LotFrontage" in all_features.columns:
        if "Neighborhood" in all_features.columns:
            lf = all_features["LotFrontage"]
            nb = all_features["Neighborhood"].astype("object")
            all_features["LotFrontage"] = lf.fillna(lf.groupby(nb).transform("median"))
        all_features["LotFrontage"] = all_features["LotFrontage"].fillna(all_features["LotFrontage"].median())

    # Numeric columns where NA generally means "does not exist" -> 0
    zero_fill_nums = [
        "GarageArea", "GarageCars",
        "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
        "BsmtFullBath", "BsmtHalfBath",
        "MasVnrArea",
    ]
    for c in zero_fill_nums:
        if c in all_features.columns:
            all_features[c] = pd.to_numeric(all_features[c], errors="coerce").fillna(0)

    # GarageYrBlt: 0 if no garage, else median
    if "GarageYrBlt" in all_features.columns:
        gy = pd.to_numeric(all_features["GarageYrBlt"], errors="coerce")
        if "GarageType" in all_features.columns:
            no_garage = all_features["GarageType"].astype("object").eq("None")
            gy = gy.fillna(gy.median())
            gy = gy.mask(no_garage, 0)
        else:
            gy = gy.fillna(gy.median())
        all_features["GarageYrBlt"] = gy

    # Remaining numeric NaNs -> median
    num_cols = all_features.select_dtypes(include=[np.number]).columns
    for c in num_cols:
        if all_features[c].isna().any():
            all_features[c] = all_features[c].fillna(all_features[c].median())

    # Remaining categorical NaNs -> mode (fallback "None")
    obj_cols = all_features.select_dtypes(include=["object"]).columns
    for c in obj_cols:
        if all_features[c].isna().any():
            mode = all_features[c].mode(dropna=True)
            all_features[c] = all_features[c].fillna(mode.iloc[0] if len(mode) else "None")

    # Feature creation
    if {"TotalBsmtSF", "1stFlrSF", "2ndFlrSF"}.issubset(all_features.columns):
        all_features["TotalSF"] = all_features["TotalBsmtSF"] + all_features["1stFlrSF"] + all_features["2ndFlrSF"]

    bath_cols = ["FullBath", "HalfBath", "BsmtFullBath", "BsmtHalfBath"]
    if set(bath_cols).issubset(all_features.columns):
        all_features["TotalBathrooms"] = (
            all_features["FullBath"]
            + 0.5 * all_features["HalfBath"]
            + all_features["BsmtFullBath"]
            + 0.5 * all_features["BsmtHalfBath"]
        )

    porch_cols = ["OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch"]
    if set(porch_cols).issubset(all_features.columns):
        all_features["TotalPorchSF"] = (
            all_features["OpenPorchSF"]
            + all_features["EnclosedPorch"]
            + all_features["3SsnPorch"]
            + all_features["ScreenPorch"]
        )

    # Ordinal encoding
    qual_map = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
    exposure_map = {"None": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4}
    finish_map = {"None": 0, "Unf": 1, "RFn": 2, "Fin": 3}
    fin_type_map = {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6}
    functional_map = {"Sal": 1, "Sev": 2, "Maj2": 3, "Maj1": 4, "Mod": 5, "Min2": 6, "Min1": 7, "Typ": 8}
    paved_map = {"N": 0, "P": 1, "Y": 2}

    ordinal_cols = {
        # Quality/Condition
        "ExterQual": qual_map,
        "ExterCond": qual_map,
        "BsmtQual": qual_map,
        "BsmtCond": qual_map,
        "HeatingQC": qual_map,
        "KitchenQual": qual_map,
        "FireplaceQu": qual_map,
        "GarageQual": qual_map,
        "GarageCond": qual_map,
        "PoolQC": qual_map,
        # Other ordered
        "BsmtExposure": exposure_map,
        "GarageFinish": finish_map,
        "BsmtFinType1": fin_type_map,
        "BsmtFinType2": fin_type_map,
        "Functional": functional_map,
        "PavedDrive": paved_map,
    }

    for c, mapping in ordinal_cols.items():
        if c in all_features.columns:
            all_features[c] = all_features[c].astype("object").map(mapping).fillna(0).astype(float)

    # One-hot encode remaining categoricals
    all_features = pd.get_dummies(all_features, columns=all_features.select_dtypes(include=["object"]).columns, drop_first=False)

    # Drop ID if present
    if "Id" in all_features.columns:
        all_features = all_features.drop(columns=["Id"], errors="ignore")

    # Split back
    X_all = all_features.iloc[: len(train_features)].copy()
    X_test = all_features.iloc[len(train_features) :].copy()

    # Optional correlation-based filtering (train only)
    if corr_threshold is not None and corr_threshold > 0:
        # Use only numeric features (after encoding it's all numeric)
        corr = X_all.corrwith(y)
        keep = corr.index[corr.abs() >= corr_threshold]
        X_all = X_all[keep]
        X_test = X_test[keep]

    return X_all, y, X_test


# Build features
X, y, X_test = preprocess_features(train_df, test_df, corr_threshold=0.01)

# Split into 80/20 portion (kept constant across experiments)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Log Data Setup to DagsHub
with mlflow.start_run(run_name="Final_Data_Prep"):
     mlflow.log_param("train_shape", str(X_train.shape))
     mlflow.log_param("val_shape", str(X_val.shape))
     mlflow.log_param("corr_threshold", 0.01)
     print(f"Setup Complete: Training on {X_train.shape[0]} houses.")

Setup Complete: Training on 1168 houses.
🏃 View run Final_Data_Prep at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/1a66bf2a45f8443db70fb210b4348cc2
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


## Feature Selection Experiments
In this section I implement and compare four distinct feature selection strategies to identify the most predictive variables for the model. By testing Random Forest Importance, Permutation Importance and Mutual Information against a baseline of all numeric features I can reduce dataset noise, prevent overfitting and ensure the model focuses on the most impactful data points.

In [189]:
from sklearn.feature_selection import SelectFromModel, mutual_info_regression, SelectKBest
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestRegressor
import numpy as np

# Baseline: All Features
X_train_all, X_val_all = X_train.copy(), X_val.copy()

# RF Importance Filter
selector = SelectFromModel(RandomForestRegressor(n_estimators=100, random_state=42), max_features=44)
X_train_rf = selector.fit_transform(X_train, y_train)
X_val_rf = selector.transform(X_val)

# Permutation Importance
rf_for_perm = RandomForestRegressor(n_estimators=50, random_state=42).fit(X_train, y_train)
perm_imp = permutation_importance(rf_for_perm, X_val, y_val, n_repeats=10, random_state=42)
sorted_idx = perm_imp.importances_mean.argsort()[-45:]
X_train_perm = X_train.iloc[:, sorted_idx]
X_val_perm = X_val.iloc[:, sorted_idx]

# Mutual Information
mi_selector = SelectKBest(score_func=mutual_info_regression, k=45)
X_train_mi = mi_selector.fit_transform(X_train, y_train)
X_val_mi = mi_selector.transform(X_val)


## Evaluating Feature Selection Methods
This loop iterates through each version of the training and validation sets created in the previous step (Baseline, RF Filter, Permutation and Mutual Information). By training a standardized XGBoost model on each specific subset and logging the performance metrics (RMSLE, RMSE, MAE) to DagsHub I can objectively determine which feature selection technique provides the most accurate and generalizable results for this dataset.

In [190]:
import xgboost as xgb

# Create a list of our different data versions from the previous cell
data_approaches = [
    (X_train_all, X_val_all, "All_Features"),
    (X_train_rf, X_val_rf, "RF_Filter_44"),
    (X_train_perm, X_val_perm, "Permutation_45"),
    (X_train_mi, X_val_mi, "Mutual_Info_45")
]

# Test them all using a standard XGBoost model
for train_set, val_set, fs_name in data_approaches:
    # We temporarily point our global X_train/X_val to the specific version
    X_train, X_val = train_set, val_set

    # Define a standard model to keep the test fair
    test_model = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=5, random_state=42)

    # Train and log it
    train_and_log(test_model, f"Selection_Test_{fs_name}")

# Reset X_train/X_val to the All_Features baseline for now
X_train, X_val = X_train_all, X_val_all
print("Stage 1: Feature Selection testing complete. Check DagsHub!")

Registered model 'Selection_Test_All_Features' already exists. Creating a new version of this model...
Created version '5' of model 'Selection_Test_All_Features'.


Model logged: Selection_Test_All_Features
   RMSLE:          0.1382
   RMSE:           $25,959
   MAE:            $16,003
   R2 Score:       0.9121
------------------------------
🏃 View run Selection_Test_All_Features at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/37c346e8ee554077a7aecd48b41900e5
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


Registered model 'Selection_Test_RF_Filter_44' already exists. Creating a new version of this model...
Created version '5' of model 'Selection_Test_RF_Filter_44'.


Model logged: Selection_Test_RF_Filter_44
   RMSLE:          0.1545
   RMSE:           $27,704
   MAE:            $17,976
   R2 Score:       0.8999
------------------------------
🏃 View run Selection_Test_RF_Filter_44 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/48074c09c2374e5d930097cdcb814bcf
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


Registered model 'Selection_Test_Permutation_45' already exists. Creating a new version of this model...
Created version '5' of model 'Selection_Test_Permutation_45'.


Model logged: Selection_Test_Permutation_45
   RMSLE:          0.1373
   RMSE:           $25,627
   MAE:            $15,959
   R2 Score:       0.9144
------------------------------
🏃 View run Selection_Test_Permutation_45 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/c2cdde805a5f4012b979c5df5f65d7a6
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


Registered model 'Selection_Test_Mutual_Info_45' already exists. Creating a new version of this model...
Created version '5' of model 'Selection_Test_Mutual_Info_45'.


Model logged: Selection_Test_Mutual_Info_45
   RMSLE:          0.1448
   RMSE:           $26,543
   MAE:            $16,722
   R2 Score:       0.9081
------------------------------
🏃 View run Selection_Test_Mutual_Info_45 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/4ac3ccee23554ae5af1b57921005ebbf
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1
Stage 1: Feature Selection testing complete. Check DagsHub!


## Model Training
In this section I train multiple algorithms with various hyperparameter configurations and log every run to MLflow. The goal is to systematically compare underfitting (Linear/Ridge), overfitting (Decision Tree depth=None), and well-regularized models (Random Forest, XGBoost, GradientBoosting).

#### Linear Regression

In [191]:
lr_model = train_and_log(LinearRegression(), "Linear_Regression")

Registered model 'Linear_Regression' already exists. Creating a new version of this model...
Created version '13' of model 'Linear_Regression'.


Model logged: Linear_Regression
   RMSLE:          0.7236
   RMSE:           $34,441
   MAE:            $21,075
   R2 Score:       0.8454
------------------------------
🏃 View run Linear_Regression at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/4f4f9c21a9f4478fa2e6233de438a8b4
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


#### Ridge Regression

In [200]:
# Testing multiple hyperparameters for Ridge
ridge_alphas = [0.1, 1.0, 10.0]

for a in ridge_alphas:
    model_instance = Ridge(alpha=a)
    display_name = f"Ridge_alpha_{a}"
    train_and_log(model_instance, display_name)

Registered model 'Ridge_alpha_0.1' already exists. Creating a new version of this model...
Created version '5' of model 'Ridge_alpha_0.1'.


Model logged: Ridge_alpha_0.1
   RMSLE:          0.7220
   RMSE:           $32,757
   MAE:            $20,775
   R2 Score:       0.8601
------------------------------
🏃 View run Ridge_alpha_0.1 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/ec72628953c440f791f2eaafa2d949b8
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


Registered model 'Ridge_alpha_1.0' already exists. Creating a new version of this model...
Created version '5' of model 'Ridge_alpha_1.0'.


Model logged: Ridge_alpha_1.0
   RMSLE:          0.1625
   RMSE:           $31,356
   MAE:            $20,003
   R2 Score:       0.8718
------------------------------
🏃 View run Ridge_alpha_1.0 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/b4bc896ca9cb46b38b74276d3a58c3f7
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


Registered model 'Ridge_alpha_10.0' already exists. Creating a new version of this model...
Created version '5' of model 'Ridge_alpha_10.0'.


Model logged: Ridge_alpha_10.0
   RMSLE:          0.1603
   RMSE:           $31,629
   MAE:            $19,168
   R2 Score:       0.8696
------------------------------
🏃 View run Ridge_alpha_10.0 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/359cd582e75a466e9b8b00f62172f610
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


#### Decision Tree

In [193]:
# Testing multiple hyperparameters for Decision Tree
dt_depths = [3, 5, 10, 20, None]

for depth in dt_depths:
    model_instance = DecisionTreeRegressor(max_depth=depth, random_state=42)
    display_name = f"DecisionTree_depth_{'None' if depth is None else depth}"
    train_and_log(model_instance, display_name)

Registered model 'DecisionTree_depth_3' already exists. Creating a new version of this model...
Created version '6' of model 'DecisionTree_depth_3'.


Model logged: DecisionTree_depth_3
   RMSLE:          0.2275
   RMSE:           $40,352
   MAE:            $28,695
   R2 Score:       0.7877
------------------------------
🏃 View run DecisionTree_depth_3 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/f7b2e66e7eb54156a5c4bebf2f550b68
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


Registered model 'DecisionTree_depth_5' already exists. Creating a new version of this model...
Created version '6' of model 'DecisionTree_depth_5'.


Model logged: DecisionTree_depth_5
   RMSLE:          0.1962
   RMSE:           $37,377
   MAE:            $24,312
   R2 Score:       0.8179
------------------------------
🏃 View run DecisionTree_depth_5 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/f02266d7aee142c68e6554fdcff98289
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


Registered model 'DecisionTree_depth_10' already exists. Creating a new version of this model...
Created version '6' of model 'DecisionTree_depth_10'.


Model logged: DecisionTree_depth_10
   RMSLE:          0.2086
   RMSE:           $39,268
   MAE:            $25,562
   R2 Score:       0.7990
------------------------------
🏃 View run DecisionTree_depth_10 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/1e6a8bb9bd6045a78b3699e60a941e8c
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


Registered model 'DecisionTree_depth_20' already exists. Creating a new version of this model...
Created version '6' of model 'DecisionTree_depth_20'.


Model logged: DecisionTree_depth_20
   RMSLE:          0.2034
   RMSE:           $39,765
   MAE:            $25,511
   R2 Score:       0.7938
------------------------------
🏃 View run DecisionTree_depth_20 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/33e962e874e64eee8abcb549fd4a6a22
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


Registered model 'DecisionTree_depth_None' already exists. Creating a new version of this model...
Created version '6' of model 'DecisionTree_depth_None'.


Model logged: DecisionTree_depth_None
   RMSLE:          0.2022
   RMSE:           $39,811
   MAE:            $25,564
   R2 Score:       0.7934
------------------------------
🏃 View run DecisionTree_depth_None at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/eea65d8031f7435cbf8f261e00d16300
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


#### Random Forest

In [194]:
# Testing multiple hyperparameters for Random Forest
rf_options = [50, 100, 200]

for n in rf_options:
    model_instance = RandomForestRegressor(n_estimators=n, random_state=42)
    display_name = f"RandomForest_est_{n}"
    train_and_log(model_instance, display_name)

Registered model 'RandomForest_est_50' already exists. Creating a new version of this model...
Created version '7' of model 'RandomForest_est_50'.


Model logged: RandomForest_est_50
   RMSLE:          0.1519
   RMSE:           $30,640
   MAE:            $18,063
   R2 Score:       0.8776
------------------------------
🏃 View run RandomForest_est_50 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/8aced19d2388409683a5b87aef5fb0ce
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


Registered model 'RandomForest_est_100' already exists. Creating a new version of this model...
Created version '7' of model 'RandomForest_est_100'.


Model logged: RandomForest_est_100
   RMSLE:          0.1514
   RMSE:           $29,925
   MAE:            $17,901
   R2 Score:       0.8832
------------------------------
🏃 View run RandomForest_est_100 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/532d65586c074fb7b1d44e5f1d7f714a
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


Registered model 'RandomForest_est_200' already exists. Creating a new version of this model...
Created version '6' of model 'RandomForest_est_200'.


Model logged: RandomForest_est_200
   RMSLE:          0.1502
   RMSE:           $29,491
   MAE:            $17,483
   R2 Score:       0.8866
------------------------------
🏃 View run RandomForest_est_200 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/6feaa3bfe85048c1a629bb127e7a3a04
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


#### Tuned Random Forest

In [195]:
# Testing multiple hyperparameters for Tuned Random Forest
rf_depths = [10, 15, 20]

for depth in rf_depths:
    model_instance = RandomForestRegressor(n_estimators=200, max_depth=depth, random_state=42)
    display_name = f"Tuned_RF_depth_{depth}"
    train_and_log(model_instance, display_name)

Registered model 'Tuned_RF_depth_10' already exists. Creating a new version of this model...
Created version '7' of model 'Tuned_RF_depth_10'.


Model logged: Tuned_RF_depth_10
   RMSLE:          0.1516
   RMSE:           $29,371
   MAE:            $17,608
   R2 Score:       0.8875
------------------------------
🏃 View run Tuned_RF_depth_10 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/9157d001778c4e718999c6c995bd44e6
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


Registered model 'Tuned_RF_depth_15' already exists. Creating a new version of this model...
Created version '6' of model 'Tuned_RF_depth_15'.


Model logged: Tuned_RF_depth_15
   RMSLE:          0.1502
   RMSE:           $29,326
   MAE:            $17,515
   R2 Score:       0.8879
------------------------------
🏃 View run Tuned_RF_depth_15 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/e028350d98ac48e397f53752877de440
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


Registered model 'Tuned_RF_depth_20' already exists. Creating a new version of this model...
Created version '6' of model 'Tuned_RF_depth_20'.


Model logged: Tuned_RF_depth_20
   RMSLE:          0.1500
   RMSE:           $29,317
   MAE:            $17,484
   R2 Score:       0.8879
------------------------------
🏃 View run Tuned_RF_depth_20 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/a8758f5674b04f108c2cccd90a66ef78
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


#### Gradient Boosting

In [196]:
# Testing multiple hyperparameters for Gradient Boosting
gbr_options = [100, 300, 500]

for n in gbr_options:
    model_instance = GradientBoostingRegressor(n_estimators=n, random_state=42)
    display_name = f"GBR_est_{n}"
    train_and_log(model_instance, display_name)

Registered model 'GBR_est_100' already exists. Creating a new version of this model...
Created version '4' of model 'GBR_est_100'.


Model logged: GBR_est_100
   RMSLE:          0.1389
   RMSE:           $27,477
   MAE:            $16,174
   R2 Score:       0.9016
------------------------------
🏃 View run GBR_est_100 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/5396282c1a554114b25e713c84ca6430
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


Registered model 'GBR_est_300' already exists. Creating a new version of this model...
Created version '4' of model 'GBR_est_300'.


Model logged: GBR_est_300
   RMSLE:          0.1383
   RMSE:           $27,187
   MAE:            $15,650
   R2 Score:       0.9036
------------------------------
🏃 View run GBR_est_300 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/51079ea653a64a8d8ad8b903a1bbe815
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


Registered model 'GBR_est_500' already exists. Creating a new version of this model...
Created version '4' of model 'GBR_est_500'.


Model logged: GBR_est_500
   RMSLE:          0.1415
   RMSE:           $27,165
   MAE:            $15,916
   R2 Score:       0.9038
------------------------------
🏃 View run GBR_est_500 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/c8a9a1b5bd504316a5a90623f46d3b9f
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


#### AdaBoost

In [198]:
# Testing multiple hyperparameters for AdaBoost
ada_options = [50, 100, 150]

for n in ada_options:
    model_instance = AdaBoostRegressor(n_estimators=n, random_state=42)
    display_name = f"AdaBoost_est_{n}"
    train_and_log(model_instance, display_name)

Registered model 'AdaBoost_est_50' already exists. Creating a new version of this model...
Created version '5' of model 'AdaBoost_est_50'.


Model logged: AdaBoost_est_50
   RMSLE:          0.2034
   RMSE:           $35,839
   MAE:            $23,735
   R2 Score:       0.8325
------------------------------
🏃 View run AdaBoost_est_50 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/b71494ee4178499cad7bc476a79a65ac
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


Registered model 'AdaBoost_est_100' already exists. Creating a new version of this model...
Created version '5' of model 'AdaBoost_est_100'.


Model logged: AdaBoost_est_100
   RMSLE:          0.2094
   RMSE:           $38,540
   MAE:            $25,068
   R2 Score:       0.8064
------------------------------
🏃 View run AdaBoost_est_100 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/4e19ad1133dd47bdb58a684d638382b1
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


Registered model 'AdaBoost_est_150' already exists. Creating a new version of this model...
Created version '4' of model 'AdaBoost_est_150'.


Model logged: AdaBoost_est_150
   RMSLE:          0.2131
   RMSE:           $39,403
   MAE:            $25,899
   R2 Score:       0.7976
------------------------------
🏃 View run AdaBoost_est_150 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/d65b7273bc1c46d998d85ad050e00f16
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


#### XGBoost (Champion)
XGBoost is tested with three learning rates to find the optimal balance between speed and accuracy. After comparing all models, XGBoost with `learning_rate=0.05` achieved the best RMSLE (0.1434) and is registered as the Champion model.

In [199]:
import xgboost as xgb

# Testing multiple hyperparameters for XGBoost
xgb_lrs = [0.01, 0.05, 0.1]

for lr in xgb_lrs:
    model_instance = xgb.XGBRegressor(n_estimators=1000, learning_rate=lr, max_depth=5)
    display_name = f"XGBoost_lr_{lr}"
    
    with mlflow.start_run(run_name=display_name):
        model_instance.fit(X_train, y_train)
        preds = model_instance.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, preds))
        mae = mean_absolute_error(y_val, preds)
        r2 = r2_score(y_val, preds)
        rmsle = np.sqrt(mean_squared_log_error(y_val, np.clip(preds, 0, None)))

        mlflow.log_param("model_type", display_name)
        mlflow.log_param("learning_rate", lr)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2_score", r2)
        mlflow.log_metric("rmsle", rmsle)

        # Register champion model simply as 'XGBoost' for easy retrieval in inference
        registered_name = "XGBoost" if lr == 0.05 else display_name
        mlflow.xgboost.log_model(
            xgb_model=model_instance,
            artifact_path="model",
            registered_model_name=registered_name
        )

        print(f"Model logged: {display_name} (registered as: {registered_name})")
        print(f"   RMSLE:          {rmsle:.4f}")
        print(f"   RMSE:           ${rmse:,.0f}")
        print(f"   MAE:            ${mae:,.0f}")
        print(f"   R2 Score:       {r2:.4f}")
        print("-" * 30)


Registered model 'XGBoost_lr_0.01' already exists. Creating a new version of this model...
Created version '7' of model 'XGBoost_lr_0.01'.


Model logged: XGBoost_lr_0.01 (registered as: XGBoost_lr_0.01)
   RMSLE:          0.1379
   RMSE:           $25,704
   MAE:            $15,864
   R2 Score:       0.9139
------------------------------
🏃 View run XGBoost_lr_0.01 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/ee997857438145c68521f4a51ba2d17e
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


Registered model 'XGBoost' already exists. Creating a new version of this model...
Created version '9' of model 'XGBoost'.


Model logged: XGBoost_lr_0.05 (registered as: XGBoost)
   RMSLE:          0.1388
   RMSE:           $25,925
   MAE:            $16,009
   R2 Score:       0.9124
------------------------------
🏃 View run XGBoost_lr_0.05 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/7c7c3b6bc19446ab8d15f1b8f2180e3a
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1


Registered model 'XGBoost_lr_0.1' already exists. Creating a new version of this model...
Created version '7' of model 'XGBoost_lr_0.1'.


Model logged: XGBoost_lr_0.1 (registered as: XGBoost_lr_0.1)
   RMSLE:          0.1376
   RMSE:           $25,767
   MAE:            $15,617
   R2 Score:       0.9134
------------------------------
🏃 View run XGBoost_lr_0.1 at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1/runs/8196e4b6cbbc4bdb9aa4620e4a9feaf2
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow/#/experiments/1
